In [1]:
import json
import pandas as pd
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=3000)
db = client["oulad_ews"]

# 1) predictions: one document per student per cutoff
col = db["predictions"]
col.drop()                                   # makes the notebook safe to rerun
with open("../data/processed/records.jsonl") as f:
    docs = [json.loads(line) for line in f]
for k in range(0, len(docs), 10000):
    col.insert_many(docs[k:k + 10000])

col.create_index([("module", 1), ("presentation", 1), ("cutoff_day", 1), ("risk_score", -1)])
col.create_index([("id_student", 1), ("cutoff_day", 1)])

# 2) model metrics: for the performance chart
db["model_metrics"].drop()
db["model_metrics"].insert_many(pd.read_csv("../reports/results_by_cutoff.csv").to_dict("records"))

# 3) checks
print("predictions stored:", col.count_documents({}))                                   # expect 79784
print("day-28 flagged:", col.count_documents({"cutoff_day": 28, "flagged": True}))       # expect 14685
print("metrics rows:", db["model_metrics"].count_documents({}))                          # expect 6
print(col.find_one({"flagged": True, "cutoff_day": 28}, {"_id": 0}))

predictions stored: 79784
day-28 flagged: 14685
metrics rows: 6
{'id_student': 106577, 'module': 'AAA', 'presentation': '2013J', 'cutoff_day': 28, 'split': 'train', 'risk_score': 0.4584, 'flagged': True, 'actual_at_risk': 0, 'theme_impacts': {'engagement': 0.102, 'assessments': -0.14, 'background': 0.439, 'course': -0.316}, 'reasons': [{'theme': 'engagement', 'impact': 0.102, 'text': '318 clicks over 12 active days (typical student: 358 clicks, 15 days)'}]}
